# Gemma 3 1B-IT Reasoning Fine-Tuning
https://www.kaggle.com/competitions/google-tunix-hackathon/writeups/gemma3-reasoning

Three-stage pipeline to improve Gemma 3 1B-IT reasoning on TPU using JAX/Tunix:

1. **SFT Stage 1** — Math & science reasoning (GSM8K + ScienceQA)
2. **SFT Stage 2** — Diverse tasks with math retention (+ MBPP, XSum, WritingPrompts)
3. **GRPO** — Reinforcement learning with verifiable math rewards

Evaluated on GSM8K test set at each stage boundary.

## Setup

In [1]:
import importlib

%pip install -q kagglehub
%pip install -q safetensors
%pip install -q tensorflow
%pip install -q tensorflow_datasets
%pip install -q tensorboardX
%pip install -q transformers
%pip install -q grain
%pip install -q datasets
%pip install -q wandb
%pip install google-tunix[prod]
%pip uninstall -q flax -y
%pip install flax==0.12.3

%pip install -q 'numpy>2'
%pip install -U transformers==4.57.1

  Using cached flax-0.12.3-py3-none-any.whl.metadata (11 kB)
Using cached flax-0.12.3-py3-none-any.whl (490 kB)


In [2]:
import wandb
wandb.init = lambda *args, **kwargs: None
wandb.log = lambda *args, **kwargs: None
wandb.finish = lambda *args, **kwargs: None

from google.colab import auth
auth.authenticate_user()

import re, random, gc, os, json, shutil
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

import jax
import jax.numpy as jnp
import optax
import qwix
from flax import nnx
from jax.sharding import NamedSharding, PartitionSpec as P, Mesh
from orbax import checkpoint as ocp
from datasets import load_dataset

from tunix.generate import sampler as sampler_lib
from tunix.models.gemma3 import params, model
from tunix.sft import peft_trainer, metrics_logger
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner
from tunix.rl.rollout import base_rollout
import tunix.sft.metrics_logger as ml

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:
# Set True to verify pipeline in ~5 min, False for real training
SMOKE_TEST = True

if SMOKE_TEST:
    SFT_STEPS_S1 = 10
    SFT_STEPS_S2 = 5
    GRPO_STEPS = 3
    EVAL_SAMPLES = 5
    BATCH_SIZE = 2
    print("SMOKE TEST MODE")
else:
    SFT_STEPS_S1 = 1000
    SFT_STEPS_S2 = 600
    GRPO_STEPS = 50
    EVAL_SAMPLES = 100
    BATCH_SIZE = 4
    est_min = (SFT_STEPS_S1 + SFT_STEPS_S2) * 3 // 60
    print(f"FULL TRAINING MODE: ~{est_min} min SFT + GRPO")

SMOKE TEST MODE


## Data

In [4]:
@dataclass
class Sample:
    prompt: str
    completion: str
    metadata: Optional[Dict[str, Any]] = field(default_factory=dict)


class Dataset:
    def __init__(self, name: str, split: str = "train"):
        self.name = name
        self.split = split
        self.samples: List[Sample] = []

    def load(self) -> None:
        raise NotImplementedError

    def __len__(self):  return len(self.samples)
    def __iter__(self): return iter(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


class GSM8KDataset(Dataset):
    def __init__(self, split="train"):
        super().__init__("gsm8k", split)

    def load(self):
        for item in load_dataset("gsm8k", "main", split=self.split):
            self.samples.append(Sample(
                prompt=f"Solve this math problem step by step:\n\n{item['question']}\n\nSolution:",
                completion=item["answer"],
                metadata={"source": "gsm8k"},
            ))


class ScienceQADataset(Dataset):
    def __init__(self, split="train"):
        super().__init__("science_qa", split)

    def load(self):
        for item in load_dataset("derek-thomas/ScienceQA", split=self.split):
            choices = item.get("choices", [])
            choices_text = "\n".join(f"{chr(65+i)}. {c}" for i, c in enumerate(choices))
            answer_idx = item.get("answer", 0)
            answer_letter = chr(65 + answer_idx) if isinstance(answer_idx, int) else "A"
            explanation = item.get("solution", "")
            completion = f"{answer_letter}\n\nExplanation: {explanation}" if explanation else answer_letter
            self.samples.append(Sample(
                prompt=f"Answer this science question:\n\n{item['question']}\n\n{choices_text}\n\nAnswer:",
                completion=completion,
                metadata={"source": "science_qa"},
            ))


class MBPPDataset(Dataset):
    def __init__(self, split="train"):
        super().__init__("mbpp", split)

    def load(self):
        for item in load_dataset("mbpp", split=self.split):
            test_str = "\n".join(item.get("test_list", [])[:3])
            self.samples.append(Sample(
                prompt=f"Write a Python function to solve this problem:\n\n{item['text']}\n\nTest cases:\n{test_str}\n\nSolution:",
                completion=item["code"],
                metadata={"source": "mbpp", "task_id": item.get("task_id")},
            ))


class XSumDataset(Dataset):
    def __init__(self, split="train"):
        super().__init__("xsum", split)

    def load(self):
        for item in load_dataset("xsum", split=self.split):
            self.samples.append(Sample(
                prompt=f"Summarize the following article in one sentence:\n\n{item['document']}\n\nSummary:",
                completion=item["summary"],
                metadata={"source": "xsum"},
            ))


class WritingPromptsDataset(Dataset):
    def __init__(self, split="train"):
        super().__init__("writing_prompts", split)

    def load(self):
        for item in load_dataset("euclaise/writingprompts", split=self.split):
            self.samples.append(Sample(
                prompt=f"Write a creative story based on this prompt:\n\n{item['prompt']}\n\nStory:",
                completion=item["story"],
                metadata={"source": "writing_prompts"},
            ))


class Sampler:
    def __init__(self, datasets: List[Dataset], weights: Optional[List[float]] = None):
        self.datasets = datasets
        self.weights = weights
        self.all_samples = [s for ds in datasets for s in ds]

    def sample(self, batch_size: int, num_batches: int):
        for _ in range(num_batches):
            if self.weights is None:
                yield random.sample(self.all_samples, min(batch_size, len(self.all_samples)))
            else:
                batch = []
                for _ in range(batch_size):
                    idx = random.choices(range(len(self.datasets)), weights=self.weights, k=1)[0]
                    if len(self.datasets[idx]) > 0:
                        batch.append(random.choice(list(self.datasets[idx])))
                yield batch

    def __len__(self): return len(self.all_samples)

In [5]:
print("Loading datasets...")

gsm8k = GSM8KDataset("train");       gsm8k.load()
science_qa = ScienceQADataset("train"); science_qa.load()
mbpp = MBPPDataset("train");          mbpp.load()
xsum = XSumDataset("train");          xsum.load()
writing = WritingPromptsDataset("train"); writing.load()
gsm8k_test = GSM8KDataset("test");    gsm8k_test.load()

for name, ds in [("GSM8K", gsm8k), ("ScienceQA", science_qa), ("MBPP", mbpp),
                  ("XSum", xsum), ("WritingPrompts", writing), ("GSM8K test", gsm8k_test)]:
    print(f"  {name}: {len(ds)} samples")

Loading datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

data/validation-00000-of-00001-137b93e1e(…):   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/272600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15138 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15620 [00:00<?, ? examples/s]

  GSM8K: 7473 samples
  ScienceQA: 12726 samples
  MBPP: 374 samples
  XSum: 204045 samples
  WritingPrompts: 272600 samples
  GSM8K test: 1319 samples


## Model

In [6]:
SAVE_PATH = "/tmp/save_path"

def save_base_checkpoint():
    """Save base Gemma3-1B-IT to disk so we can load it multiple times."""
    gemma = params.create_model_from_checkpoint(
        params.GEMMA3_1B_IT, model.ModelConfig.gemma3_1b_it()
    )
    _, state = nnx.split(gemma)
    ocp.Checkpointer(ocp.StandardCheckpointHandler()).save(
        SAVE_PATH, args=ocp.args.StandardSave(state)
    )
    del gemma, state
    gc.collect()


def load_model(ckpt_path):
    """Load model from checkpoint with bf16 and proper sharding."""
    devices = np.array(jax.devices()).reshape(1, 1)
    mesh = Mesh(devices, axis_names=("fsdp", "tp"))
    gemma_config = model.ModelConfig.gemma3_1b_it()

    abs_gemma = nnx.eval_shape(
        lambda: params.create_model_from_checkpoint(params.GEMMA3_1B_IT, gemma_config)
    )
    abs_state = nnx.state(abs_gemma)
    abs_state = jax.tree.map(
        lambda a, s: jax.ShapeDtypeStruct(a.shape, jnp.bfloat16, sharding=s),
        abs_state, nnx.get_named_sharding(abs_state, mesh),
    )

    restored = ocp.Checkpointer(ocp.StandardCheckpointHandler()).restore(
        ckpt_path, args=ocp.args.StandardRestore(item=abs_state)
    )
    graph_def, _ = nnx.split(abs_gemma)
    tokenizer = params.create_tokenizer()
    return nnx.merge(graph_def, restored), mesh, tokenizer, gemma_config


def apply_lora(base_model, mesh):
    """Apply LoRA adapters to attention and MLP layers."""
    state = nnx.state(base_model)
    nnx.update(base_model, jax.tree.map(jax.device_get, state))

    lora_model = qwix.apply_lora_to_model(
        base_model,
        qwix.LoraProvider(
            module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj|.*attn_vec_einsum",
            rank=32, alpha=32.0,
        ),
        rngs=nnx.Rngs(0),
        **base_model.get_model_input(),
    )

    with mesh:
        state = nnx.state(lora_model)
        pspecs = nnx.get_partition_spec(state)

        def safe_shard(param, pspec):
            if pspec is None or len(pspec) != len(param.shape):
                pspec = P(*([None] * len(param.shape)))
            return jax.device_put(param, NamedSharding(mesh, pspec))

        nnx.update(lora_model, jax.tree.map(safe_shard, state, pspecs))

    return lora_model

In [7]:
save_base_checkpoint()

ref_model, mesh, tokenizer, gemma_config = load_model(SAVE_PATH)
lora_base, _, _, _ = load_model(SAVE_PATH)
lora_model = apply_lora(lora_base, mesh)

## Training & Evaluation Infrastructure

In [8]:
MAX_SEQ_LEN = 1024
CAUSAL_MASK = np.tril(np.ones((1, MAX_SEQ_LEN, MAX_SEQ_LEN), dtype=np.bool_))
EOS_TOKEN_ID = 106  # <end_of_turn>, verified via tokenizer.piece_to_id


def make_batches(sampler, batch_size, num_batches):
    """Tokenize and batch samples for SFT."""
    batches = []
    for batch in sampler.sample(batch_size, num_batches):
        tokens_list, mask_list = [], []
        for sample in batch:
            tokens = tokenizer.encode(sample.prompt + sample.completion)[:MAX_SEQ_LEN]
            length = len(tokens)
            tokens_list.append(tokens + [tokenizer.pad_id()] * (MAX_SEQ_LEN - length))
            mask_list.append([1] * length + [0] * (MAX_SEQ_LEN - length))

        input_tokens = np.array(tokens_list, dtype=np.int32)
        input_mask = np.array(mask_list, dtype=np.int32)
        B, L = input_tokens.shape

        batches.append({
            "input_tokens": input_tokens,
            "input_mask": input_mask,
            "positions": np.tile(np.arange(L), (B, 1)).astype(np.int32),
            "attention_mask": np.broadcast_to(CAUSAL_MASK, (B, L, L)),
        })
    return batches


def extract_gsm8k_answer(text: str) -> Optional[float]:
    """Extract numerical answer from GSM8K-format or free-form text."""
    for pattern in [
        r'####\s*([+-]?[\d,]+\.?\d*)',                                      # #### 42
        r'(?:the\s+)?(?:final\s+)?answer\s+is[:\s]*\$?([+-]?[\d,]+\.?\d*)', # the answer is 42
        r'\\boxed\{([+-]?[\d,]+\.?\d*)\}',                                 # \boxed{42}
    ]:
        match = re.search(pattern, text, re.I)
        if match:
            return float(match.group(1).replace(',', ''))

    # Fallback: last number in text
    numbers = re.findall(r'[+-]?[\d,]+\.?\d*', text)
    return float(numbers[-1].replace(',', '')) if numbers else None


def evaluate_gsm8k(eval_model, tok, test_dataset, num_samples=100):
    """Evaluate GSM8K accuracy with near-greedy decoding."""
    gen = sampler_lib.Sampler(
        transformer=eval_model, tokenizer=tok,
        cache_config=sampler_lib.CacheConfig(
            cache_size=1536,
            num_layers=eval_model.config.num_layers,
            num_kv_heads=eval_model.config.num_kv_heads,
            head_dim=eval_model.config.head_dim,
        ),
    )

    correct, total = 0, 0
    for sample in list(test_dataset)[:num_samples]:
        prompt = f"<start_of_turn>user\n{sample.prompt}<end_of_turn>\n<start_of_turn>model\n"
        out = gen(
            input_strings=[prompt], max_generation_steps=768,
            temperature=0.01, top_k=1, top_p=1.0,
            echo=False, eos_tokens=[EOS_TOKEN_ID],
        )
        pred = extract_gsm8k_answer(out.text[0])
        gold = extract_gsm8k_answer(sample.completion)
        if pred is not None and gold is not None and abs(pred - gold) < 1e-3:
            correct += 1
        total += 1
        if total % 25 == 0:
            print(f"  [{total}/{num_samples}] acc: {correct/total:.1%}")

    acc = correct / total if total > 0 else 0.0
    print(f"GSM8K: {correct}/{total} = {acc:.2%}")
    return {"accuracy": acc, "correct": correct, "total": total}


def sft(mdl, sampler, num_steps=500, batch_size=4, learning_rate=2e-5,
        eval_every=50, stage_name="sft"):
    """Run supervised fine-tuning with LoRA via Tunix PeftTrainer."""
    print(f"{stage_name}: {num_steps} steps, bs={batch_size}, lr={learning_rate}")

    train_batches = make_batches(sampler, batch_size, num_steps)
    val_batches = make_batches(sampler, batch_size, max(num_steps // 10, 5))

    lr_schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=learning_rate,
        warmup_steps=int(num_steps * 0.1), decay_steps=num_steps,
        end_value=learning_rate * 0.01,
    )

    config = peft_trainer.TrainingConfig(
        eval_every_n_steps=eval_every, max_steps=num_steps,
        checkpoint_root_directory=f"/tmp/{stage_name}/ckpts",
        metrics_logging_options=metrics_logger.MetricsLoggerOptions(
            log_dir=f"/tmp/{stage_name}/logs", flush_every_n_steps=10,
        ),
    )

    trainer = (
        peft_trainer.PeftTrainer(mdl, optax.adamw(lr_schedule), config)
        .with_gen_model_input_fn(lambda x: x)
    )
    trainer.train(train_batches, val_batches)
    print(f"{stage_name} complete.")
    return mdl


def replicate_all(mdl, mesh):
    """Replicate all params on mesh (needed before GRPO)."""
    state = nnx.state(mdl)
    replicated = jax.tree.map(
        lambda p: jax.device_put(p, NamedSharding(mesh, P(*([None] * len(p.shape)))))
        if hasattr(p, 'shape') else p,
        state,
    )
    nnx.update(mdl, replicated)


def move_to_device(mdl, mesh):
    """Move model back to TPU after GRPO's offload_to_cpu."""
    state = nnx.state(mdl)
    on_device = jax.tree.map(
        lambda x: jax.device_put(x, NamedSharding(mesh, P(*([None] * len(x.shape)))))
        if hasattr(x, 'shape') else x,
        state,
    )
    nnx.update(mdl, on_device)

## Stage 1: SFT on Math & Science

In [9]:
print("=" * 60)
print("BASELINE")
print("=" * 60)
baseline = evaluate_gsm8k(ref_model, tokenizer, gsm8k_test, num_samples=EVAL_SAMPLES)

print("\n" + "=" * 60)
print("STAGE 1: SFT — Math & Science")
print("=" * 60)

stage1_sampler = Sampler([gsm8k, science_qa], weights=[0.7, 0.3])

lora_model = sft(
    lora_model, stage1_sampler,
    num_steps=SFT_STEPS_S1, batch_size=BATCH_SIZE,
    learning_rate=2e-5, stage_name="stage1_math",
)

print("\nPost-Stage-1:")
stage1_results = evaluate_gsm8k(lora_model, tokenizer, gsm8k_test, num_samples=EVAL_SAMPLES)

BASELINE
GSM8K: 2/5 = 40.00%

STAGE 1: SFT — Math & Science
stage1_math: 10 steps, bs=2, lr=2e-05


Training:   0%|          | 0/10 [00:00<?, ?step/s]

stage1_math complete.

Post-Stage-1:
GSM8K: 2/5 = 40.00%


## Stage 2: SFT on Diverse Tasks (with math retention)

In [10]:
print("=" * 60)
print("STAGE 2: SFT — Diverse + Math Retention")
print("=" * 60)

stage2_sampler = Sampler(
    [gsm8k, science_qa, mbpp, xsum, writing],
    weights=[0.25, 0.15, 0.25, 0.2, 0.15],
)

lora_model = sft(
    lora_model, stage2_sampler,
    num_steps=SFT_STEPS_S2, batch_size=BATCH_SIZE,
    learning_rate=1e-5, stage_name="stage2_diverse",
)

print("\nPost-Stage-2:")
stage2_results = evaluate_gsm8k(lora_model, tokenizer, gsm8k_test, num_samples=EVAL_SAMPLES)

STAGE 2: SFT — Diverse + Math Retention
stage2_diverse: 5 steps, bs=2, lr=1e-05


Training:   0%|          | 0/5 [00:00<?, ?step/s]

stage2_diverse complete.

Post-Stage-2:
GSM8K: 2/5 = 40.00%


## Stage 3: GRPO with Math Reward

In [13]:
class TokenizerWrapper:
    """Wraps sentencepiece tokenizer to handle numpy inputs from Tunix."""
    def __init__(self, sp):
        self.sp = sp

    def encode(self, text, **kw):
        if isinstance(text, np.ndarray):
            text = str(text.item()) if text.ndim == 0 else str(text[0])
        return self.sp.encode(str(text), **kw)

    def decode(self, ids, **kw):
        if isinstance(ids, np.ndarray): ids = ids.tolist()
        return self.sp.decode(ids, **kw)

    def vocab_size(self): return self.sp.get_piece_size()
    def __getattr__(self, name): return getattr(self.sp, name)


# Build answer key for reward verification
gsm8k_answer_key = {}
for s in gsm8k:
    gold = extract_gsm8k_answer(s.completion)
    if gold is not None:
        gsm8k_answer_key[s.prompt[:300]] = gold
print(f"Answer key: {len(gsm8k_answer_key)} problems")


def math_reward_fn(prompts, completions, **kwargs):
    """Reward = 1.0 for correct answer, partial credit for structure."""
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = str(completion)
        words = text.split()
        unique_ratio = len(set(words)) / max(len(words), 1)

        if unique_ratio < 0.3 or len(words) < 3:
            rewards.append(0.0)
            continue

        pred = extract_gsm8k_answer(text)
        gold = gsm8k_answer_key.get(str(prompt)[:300])
        correct = pred is not None and gold is not None and abs(pred - gold) < 1e-3

        if correct:
            rewards.append(1.0)
        else:
            r = 0.0
            if re.search(r'(step|then|therefore|because|\d+\s*[+\-*/]\s*\d+)', text, re.I):
                r += 0.2
            if pred is not None:
                r += 0.1
            if text.strip().endswith(('.', '!', '?', ')')):
                r += 0.1
            rewards.append(r)
    return rewards


def grpo(lora_model, ref_model, tokenizer, reward_fns, sampler, mesh, max_steps=50):
    """Run GRPO training with math-verifiable rewards."""
    print(f"Starting GRPO: {max_steps} steps")

    optimizer = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=optax.schedules.warmup_cosine_decay_schedule(
            init_value=0.0, peak_value=1e-6,
            warmup_steps=10, decay_steps=100, end_value=0.0,
        )),
    )

    cluster_config = rl_cluster_lib.ClusterConfig(
        role_to_mesh={
            rl_cluster_lib.Role.ACTOR: mesh,
            rl_cluster_lib.Role.REFERENCE: mesh,
            rl_cluster_lib.Role.ROLLOUT: mesh,
        },
        rollout_engine="vanilla",
        offload_to_cpu=True,
        training_config=rl_cluster_lib.RLTrainingConfig(
            actor_optimizer=optimizer, max_steps=max_steps,
            mini_batch_size=2, train_micro_batch_size=1,
            rollout_micro_batch_size=2, compute_logps_micro_batch_size=1,
            checkpoint_root_directory="/tmp/grpo/ckpts",
            eval_every_n_steps=10,
            metrics_logging_options=metrics_logger.MetricsLoggerOptions(
                log_dir="/tmp/grpo/logs", flush_every_n_steps=10,
            ),
        ),
        rollout_config=base_rollout.RolloutConfig(
            max_tokens_to_generate=512, max_prompt_length=256,
            kv_cache_size=1024, temperature=0.8, top_p=0.95, top_k=50,
        ),
    )

    rl_cluster = rl_cluster_lib.RLCluster(
        actor=lora_model, reference=ref_model,
        tokenizer=TokenizerWrapper(tokenizer),
        cluster_config=cluster_config,
    )

    # Workaround: skip optimizer sharding on single-device mesh
    rl_cluster.actor_trainer._shard_optimizer = lambda mesh: None

    # Workaround: guard against string-type metric logging crashes
    original_log = ml.MetricsLogger.log
    def safe_log(self, *args, **kwargs):
        for i, arg in enumerate(list(args)):
            try:
                if isinstance(arg, str) and i > 0: float(arg)
            except ValueError: continue
            try:
                if hasattr(arg, 'dtype') and 'U' in str(arg.dtype): return
            except (TypeError, AttributeError): pass
        original_log(self, *args, **kwargs)
    ml.MetricsLogger.log = safe_log

    learner = GRPOLearner(
        rl_cluster=rl_cluster,
        reward_fns=reward_fns,
        algo_config=GRPOConfig(num_generations=4, num_iterations=2, beta=0.1, epsilon=0.2),
    )

    def prompt_batches():
        for batch in sampler.sample(batch_size=2, num_batches=100):
            yield {"prompts": [str(s.prompt)[:300] for s in batch]}

    learner.train(prompt_batches())
    return rl_cluster.actor_trainer.model

Answer key: 7473 problems


In [14]:
print("=" * 60)
print("STAGE 3: GRPO — Math RL")
print("=" * 60)

grpo_sampler = Sampler([gsm8k])

replicate_all(lora_model, mesh)
replicate_all(ref_model, mesh)

lora_model = grpo(
    lora_model, ref_model, tokenizer,
    reward_fns=[math_reward_fn],
    sampler=grpo_sampler, mesh=mesh, max_steps=GRPO_STEPS,
)

move_to_device(lora_model, mesh)

print("\nPost-Stage-3:")
stage3_results = evaluate_gsm8k(lora_model, tokenizer, gsm8k_test, num_samples=EVAL_SAMPLES)

STAGE 3: GRPO — Math RL
Starting GRPO: 3 steps


Actor Training:   0%|          | 0/3 [00:00<?, ?step/s]


Post-Stage-3:
GSM8K: 2/5 = 40.00%


## Results

In [15]:
print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"  {'Stage':<30} {'GSM8K':>10}")
print(f"  {'-'*30} {'-'*10}")
print(f"  {'Baseline':<30} {baseline['accuracy']:>10.2%}")
print(f"  {'+ Stage 1 SFT (math)':<30} {stage1_results['accuracy']:>10.2%}")
print(f"  {'+ Stage 2 SFT (diverse)':<30} {stage2_results['accuracy']:>10.2%}")
print(f"  {'+ Stage 3 GRPO':<30} {stage3_results['accuracy']:>10.2%}")
print("=" * 60)

RESULTS
  Stage                               GSM8K
  ------------------------------ ----------
  Baseline                           40.00%
  + Stage 1 SFT (math)               40.00%
  + Stage 2 SFT (diverse)            40.00%
  + Stage 3 GRPO                     40.00%


## Inference Demo

In [16]:
gen_sampler = sampler_lib.Sampler(
    transformer=lora_model, tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=1536,
        num_layers=lora_model.config.num_layers,
        num_kv_heads=lora_model.config.num_kv_heads,
        head_dim=lora_model.config.head_dim,
    ),
)

questions = [
    "What is 25 * 13? Think step by step.",
    "If a train travels 60 km/h for 2.5 hours then 90 km/h for 1.5 hours, what is the total distance?",
    "Which is larger: 3/7 or 5/12? Explain your reasoning.",
]

for q in questions:
    prompt = f"<start_of_turn>user\n{q}<end_of_turn>\n<start_of_turn>model\n"
    out = gen_sampler(
        input_strings=[prompt], max_generation_steps=512,
        temperature=0.7, top_k=50, top_p=0.95,
        echo=False, eos_tokens=[EOS_TOKEN_ID],
    )
    print(f"\nQ: {q}")
    print(f"A: {out.text[0]}")
    print("─" * 60)


Q: What is 25 * 13? Think step by step.
A: Okay, let's calculate 25 * 13 step-by-step:

1. **Multiply 25 by 10:** 25 * 10 = 250
2. **Add the result to 13:** 250 + 13 = 263

Therefore, 25 * 13 = 263 


────────────────────────────────────────────────────────────

Q: If a train travels 60 km/h for 2.5 hours then 90 km/h for 1.5 hours, what is the total distance?
A: Let $d_1$ be the distance traveled in the first part of the journey, and $d_2$ be the distance traveled in the second part of the journey.
The speed in the first part of the journey is $v_1 = 60$ km/h, and the time is $t_1 = 2.5$ hours.
The distance traveled in the first part is $d_1 = v_1 \times t_1 = 60 \times 2.5 = 150$ km.
The speed in the second part of the journey is $v_2 = 90$ km/h, and the time is $t_2 = 1.5$ hours.
The distance traveled in the second part is $d_2 = v_2 \times t_2 = 90 \times 1.5 = 135$ km.
The total distance is $d = d_1 + d_2 = 150 + 135 = 285$ km.

Final Answer: The final answer is $\boxed{285}$
───

## Save & Export

In [17]:
from safetensors.numpy import save_file as save_safetensors

ADAPTER_DIR = "/tmp/gemma3-1b-reasoning-lora"
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Extract LoRA parameters only
state = nnx.state(lora_model)
lora_params, base_count, lora_count = {}, 0, 0

for path, var in state.flat_state():
    path_str = '.'.join(str(p) for p in path)
    arr = np.array(var[...])
    if 'lora_a' in path_str or 'lora_b' in path_str:
        lora_params[path_str] = arr
        lora_count += arr.size
    else:
        base_count += arr.size

print(f"Base: {base_count:,} params | LoRA: {lora_count:,} params ({lora_count/(base_count+lora_count)*100:.2f}%)")
save_safetensors(lora_params, os.path.join(ADAPTER_DIR, "adapter_model.safetensors"))

# Adapter config
adapter_config = {
    "base_model": "google/gemma-3-1b-it",
    "lora_rank": 32,
    "lora_alpha": 32.0,
    "target_modules": ["q_einsum", "kv_einsum", "gate_proj", "down_proj", "up_proj", "attn_vec_einsum"],
    "framework": "jax-tunix-qwix",
    "max_seq_len": MAX_SEQ_LEN,
    "results": {
        "gsm8k_baseline": baseline["accuracy"],
        "gsm8k_stage1_sft": stage1_results["accuracy"],
        "gsm8k_stage2_sft": stage2_results["accuracy"],
        "gsm8k_stage3_grpo": stage3_results["accuracy"],
    },
}
with open(os.path.join(ADAPTER_DIR, "adapter_config.json"), "w") as f:
    json.dump(adapter_config, f, indent=2)

# Model card
model_card = f"""---
base_model: google/gemma-3-1b-it
library_name: tunix
license: gemma
tags: [gemma3, lora, reasoning, math, grpo, jax, tpu]
---

# Gemma 3 1B-IT — Reasoning LoRA

LoRA adapter fine-tuned on reasoning tasks using [Tunix](https://github.com/google/tunix) on TPU v5e.

## Results (GSM8K)

| Stage | Accuracy |
|-------|----------|
| Baseline | {baseline['accuracy']:.1%} |
| + SFT Stage 1 (Math & Science) | {stage1_results['accuracy']:.1%} |
| + SFT Stage 2 (Diverse) | {stage2_results['accuracy']:.1%} |
| + GRPO (Math RL) | {stage3_results['accuracy']:.1%} |

## Training

- **LoRA**: rank=32, alpha=32, targets: q/kv_einsum, gate/down/up_proj, attn_vec_einsum
- **Stage 1**: SFT on GSM8K (70%) + ScienceQA (30%), lr=2e-5
- **Stage 2**: SFT on 5 datasets with math retention, lr=1e-5
- **Stage 3**: GRPO with verifiable math reward (correctness + structure)
- **Hardware**: TPU v5e, seq_len=1024, JAX + Tunix + Qwix
"""
with open(os.path.join(ADAPTER_DIR, "README.md"), "w") as f:
    f.write(model_card)

print(f"Saved to {ADAPTER_DIR}: {os.listdir(ADAPTER_DIR)}")

Base: 999,885,952 params | LoRA: 25,133,056 params (2.45%)
Saved to /tmp/gemma3-1b-reasoning-lora: ['README.md', 'adapter_model.safetensors', 'adapter_config.json']


In [18]:
# Download as zip
from google.colab import files
shutil.make_archive("/tmp/gemma3-1b-reasoning-lora", "zip", ADAPTER_DIR)
files.download("/tmp/gemma3-1b-reasoning-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>